# K5: BoW (TF) + SVM sa linearnim kernelom (Disneyland)
**Tip:** profesorkin drugi primer zadatka. Reprezentacija: Bag of Words, TF metrika. Klasifikator: SVM, linearni kernel. Uz tekst ide kategorijska kolona.

### Zadatak
U fajlu `Disneyland_reviews_for_class.csv` su komentari posetilaca Disneyland parkova (`Review_ID`, `Rating`, `Year_Month`, `Reviewer_Location`, `Review_Text`, `Branch`).
- Od kolone `Rating` napraviti `sentiment` (1 i 2 negativan, 3 neutralan, 4 i 5 pozitivan).
- Transformisati `Review_Text` primenom **Bag of Words modela i TF metrike**. Proceniti da li još neku varijablu treba uključiti i obrazložiti.
- Prvi model: **SVM sa linearnim kernelom** i podrazumevanim hiperparametrima.
- Metrike po klasama i ukupno, interpretacija u kontekstu zadatka.
- Drugi model: komentar sa opsezima, pa dva hiperparametra (jedan za transformaciju teksta, jedan za klasifikator) kroz 5-fold kros-validaciju.
- Metrike za podešeni model, poređenje i komentar.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import RobustScaler, OneHotEncoder, MinMaxScaler, FunctionTransformer
from sklearn import metrics
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, balanced_accuracy_score
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline   # imblearn Pipeline radi i bez SMOTE-a, pa moze da ostane uvek

from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier

import matplotlib.pyplot as plt
import seaborn as sb

from pathlib import Path
from IPython.display import display

import spacy

RAND_STATE = 1
DATA_DIR = Path.cwd().parent / 'data'

### Šta je urađeno
- Učitani podaci (nema nedostajućih vrednosti).
- Napravljena izlazna varijabla `sentiment`. Klase su jako neravnomerne: pozitivan oko 80%, neutralan 12%, negativan 8%.
- Iz `Year_Month` izdvojen mesec i pogledan kdeplot po sentimentu.
- `Branch` (3 parka) pregledan countplot-om, `Reviewer_Location` ima 86 vrednosti.
- Odluka: ulaz su `Review_Text` i `Branch` (kroz one-hot encoding u `ColumnTransformer`-u). Mesec i lokacija ne ulaze.
- Podela 80:20 sa `stratify`.

In [ ]:
data = pd.read_csv(DATA_DIR / 'Disneyland_reviews_for_class.csv')
data.info()

In [ ]:
def get_sentiment(rating: int) -> str:
    if rating == 3:
        return 'neutralan'
    return 'pozitivan' if rating > 3 else 'negativan'

data['sentiment'] = data['Rating'].apply(get_sentiment)
print(data['sentiment'].value_counts(normalize=True))

In [ ]:
data['month'] = pd.to_numeric(data['Year_Month'].str.split('-').str[1], errors='coerce')

plt.figure(figsize=(10, 6))
sb.kdeplot(data=data, x='month', hue='sentiment',
           fill=True, common_norm=False, alpha=0.5, linewidth=2)
plt.title('Raspodela meseca posete po sentimentu')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

In [ ]:
print(data[['Branch', 'Reviewer_Location']].nunique())
print(data['Branch'].value_counts())

plt.figure(figsize=(10, 6))
sb.countplot(data=data, x='Branch', hue='sentiment', fill=True, alpha=0.5)
plt.title('Broj komentara po parku i sentimentu')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

print(pd.crosstab(data['Branch'], data['sentiment'], normalize='index').round(3))

In [ ]:
X = data[['Review_Text', 'Branch']]
y = data['sentiment']
y_encodings, y_levels = pd.factorize(y)
print(list(y_levels))

X_train, X_test, y_train, y_test = train_test_split(X, y_encodings,
                                                    test_size=0.2,
                                                    random_state=RAND_STATE,
                                                    stratify=y_encodings)
print(X_train.shape, X_test.shape)

---
# ▶ OVDE KREĆEŠ

Sve iznad je gotovo. Od ove ćelije pa nadalje kucaš sam: kod, komentare u markdown ćelijama i tumačenja.

**Tvoji koraci:**

1. spaCy `en_core_web_sm` i `spacy_srediTekst`
2. `CountVectorizer(tokenizer=..., token_pattern=None, ...)` i `OneHotEncoder` za `Branch`, oba u `ColumnTransformer`
3. `SVC(kernel='linear', random_state=RAND_STATE)` i `Pipeline`, pa `fit`
4. Matrica konfuzije, `classification_report`, balanced accuracy, tumačenje (accuracy uporedi sa udelom pozitivnih od 80%)
5. Komentar o hiperparametrima, `GridSearchCV` (npr. `min_df` + `C`, `cv=5`)
6. Podešeni model i poređenje

---
## Dati kod
Ovo dobijaš uz zadatak. Pokreni ćeliju kad stigneš do poređenja modela.

In [ ]:
# DATI KOD (dobija se na kolokvijumu): poredjenje modela
def get_model_metrics(y_true: np.ndarray, y_pred: np.ndarray, model_name: str):
    report = classification_report(y_true, y_pred, output_dict=True)
    macro_avg = report['macro avg']
    weighted_avg = report['weighted avg']
    return {
        'Model': model_name,
        'Macro Precision': macro_avg['precision'],
        'Macro Recall': macro_avg['recall'],
        'Macro F1-Score': macro_avg['f1-score'],
        'Weighted Precision': weighted_avg['precision'],
        'Weighted Recall': weighted_avg['recall'],
        'Weighted F1-Score': weighted_avg['f1-score'],
        'Balanced Accuracy': balanced_accuracy_score(y_true, y_pred)
    }